<h1 align="center" style="font-size:50px;">
<b>Setup y Configuración - LLMOps</b>
</h1>

Este notebook te va a guiar paso a paso en la configuración de todo el entorno necesario para ejecutar el pipeline de LLMOps.

Vamos a cubrir:
- Configuración de Google Cloud Platform (GCP)
- Autenticación y permisos
- Habilitación de APIs necesarias
- Configuración de LangSmith para monitoring
- Verificación de que todo funcione correctamente

**Importante:** Ejecutá todas las celdas en orden. Si alguna falla, revisá el mensaje de error antes de continuar.

## **0) Pre-requisitos**

Antes de comenzar, asegurate de tener:

1. **Una cuenta de Google Cloud Platform**
   - Si no tenés una, creala en: https://console.cloud.google.com/
   - Incluye $300 USD de crédito gratuito por 90 días

2. **Tarjeta de crédito/débito vinculada**
   - Necesaria para verificar la cuenta (incluso para tier gratuito)
   - Los datasets públicos de BigQuery NO generan cargos

3. **Proyecto de GCP creado**
   - Si todavía no lo hiciste, lo creamos en la siguiente sección

## **1) Instalación de dependencias**

In [ ]:
# Instalamos todas las librerías necesarias
!pip install -q google-cloud-aiplatform
!pip install -q google-cloud-bigquery
!pip install -q google-genai
!pip install -q kfp
!pip install -q langchain langchain-openai
!pip install -q langsmith
!pip install -q pandas numpy tqdm

print("✅ Todas las dependencias fueron instaladas correctamente")

## **2) Autenticación en Google Cloud**

Según donde estés ejecutando este notebook, vas a necesitar autenticarte de diferentes maneras:

- **Google Colab**: Usá la celda de autenticación automática
- **Jupyter local**: Necesitás tener instalado y configurado Google Cloud SDK

Ejecutá la celda que corresponda a tu entorno:

In [ ]:
# SOLO PARA GOOGLE COLAB
# Si estás en Colab, ejecutá esta celda. Si no, saltala.

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Autenticación exitosa en Google Colab")
except ImportError:
    print("ℹ️ No estás en Google Colab. Usá la celda siguiente para ambiente local.")

In [ ]:
# SOLO PARA JUPYTER LOCAL
# Si ya ejecutaste la autenticación de Colab, saltá esta celda

# Verificamos si el gcloud SDK está instalado
import subprocess

try:
    result = subprocess.run(['gcloud', '--version'], capture_output=True, text=True)
    print("✅ Google Cloud SDK detectado:")
    print(result.stdout)
    print("\nSi no te autenticaste todavía, ejecutá en una terminal:")
    print("  gcloud auth login")
    print("  gcloud auth application-default login")
except FileNotFoundError:
    print("❌ Google Cloud SDK no está instalado.")
    print("Instalalo desde: https://cloud.google.com/sdk/docs/install")

## **3) Configuración del proyecto de GCP**

Ahora vamos a configurar el proyecto de Google Cloud donde vamos a trabajar.

**Si ya tenés un proyecto creado**, poné su ID en la siguiente celda.

**Si no tenés un proyecto**, crealo desde la consola: https://console.cloud.google.com/projectcreate

In [ ]:
# CONFIGURÁ TU PROJECT_ID ACÁ
# Si ya creaste el proyecto 'llmops-tutorial', dejá este valor
# Si no, reemplazalo con tu PROJECT_ID

PROJECT_ID = "llmops-tutorial"  # 👈 CAMBIAR si tu proyecto tiene otro nombre
LOCATION = "us-central1"  # Región por defecto

# Configuramos el proyecto en gcloud
print(f"Configurando proyecto: {PROJECT_ID}...")
!gcloud config set project {PROJECT_ID}

# Verificamos que el proyecto existe
import subprocess
try:
    result = subprocess.run(
        ['gcloud', 'projects', 'describe', PROJECT_ID],
        capture_output=True,
        text=True,
        timeout=10
    )

    if result.returncode == 0:
        print(f"✅ Proyecto configurado correctamente: {PROJECT_ID}")
        print(f"✅ Región configurada: {LOCATION}")
    else:
        print(f"❌ El proyecto '{PROJECT_ID}' no existe o no tenés acceso")
        print(f"\nCrealo en: https://console.cloud.google.com/projectcreate")
        print(f"O verificá el PROJECT_ID en: https://console.cloud.google.com/")
except Exception as e:
    print(f"⚠️ No se pudo verificar el proyecto (continuando de todas formas)")
    print(f"✅ Usando proyecto: {PROJECT_ID}")
    print(f"✅ Región: {LOCATION}")

## **4) Verificación de facturación**

Para usar Vertex AI y otras APIs, necesitás tener la facturación habilitada en tu proyecto.

**No te preocupes**: Los $300 USD de crédito gratuito cubren todo el uso de este tutorial.

Verificá tu estado de facturación en: https://console.cloud.google.com/billing

In [ ]:
# Verificamos si la facturación está habilitada
import subprocess

try:
    result = subprocess.run(
        ['gcloud', 'beta', 'billing', 'projects', 'describe', PROJECT_ID],
        capture_output=True,
        text=True
    )

    if 'billingEnabled: true' in result.stdout:
        print("✅ Facturación habilitada correctamente")
    else:
        print("⚠️ La facturación no está habilitada")
        print("Habilitala en: https://console.cloud.google.com/billing")
except Exception as e:
    print("ℹ️ No se pudo verificar el estado de facturación automáticamente")
    print("Por favor, verificá manualmente en: https://console.cloud.google.com/billing")

## **5) Habilitación de APIs necesarias**

Para que el pipeline de LLMOps funcione, necesitamos habilitar varias APIs de Google Cloud:

- **BigQuery API**: Para consultar el dataset de StackOverflow
- **Vertex AI API**: Para el pipeline de ML y versionado de prompts
- **Cloud Storage API**: Para guardar artefactos
- **Compute Engine API**: Para ejecutar los pipelines

Esto puede tardar 1-2 minutos.

In [ ]:
# Habilitamos todas las APIs necesarias
print("Habilitando APIs... Esto puede tardar un par de minutos.\n")

!gcloud services enable bigquery.googleapis.com --project={PROJECT_ID}
print("✅ BigQuery API habilitada")

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}
print("✅ Vertex AI API habilitada")

!gcloud services enable storage.googleapis.com --project={PROJECT_ID}
print("✅ Cloud Storage API habilitada")

!gcloud services enable compute.googleapis.com --project={PROJECT_ID}
print("✅ Compute Engine API habilitada")

!gcloud services enable notebooks.googleapis.com --project={PROJECT_ID}
print("✅ Notebooks API habilitada")

print("\n✅ Todas las APIs fueron habilitadas correctamente")

## **6) Inicialización de clientes de GCP**

In [ ]:
# Importamos las librerías necesarias
from google.cloud import aiplatform, bigquery
import pandas as pd
from datetime import datetime

# Inicializamos los clientes
aiplatform.init(project=PROJECT_ID, location=LOCATION)
bq_client = bigquery.Client(project=PROJECT_ID)

# Generamos un UID único para esta sesión
UID = datetime.now().strftime("%m%d%H%M")

print(f"✅ Cliente de Vertex AI inicializado")
print(f"✅ Cliente de BigQuery inicializado")
print(f"✅ Session UID: {UID}")

## **7) Verificación de BigQuery**

Vamos a hacer una consulta de prueba al dataset público de StackOverflow para verificar que BigQuery funciona correctamente.

In [ ]:
# Query de prueba - traemos solo 5 preguntas
test_query = """
SELECT q.id, q.title, q.score, q.view_count
FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
WHERE Score > 100
ORDER BY View_Count DESC
LIMIT 5
"""

print("Ejecutando query de prueba en BigQuery...\n")

try:
    query_job = bq_client.query(test_query)
    df_test = query_job.to_dataframe()

    print("✅ BigQuery funcionando correctamente\n")
    print("Preguntas más vistas de StackOverflow (con score > 100):\n")
    print(df_test.to_string(index=False))

except Exception as e:
    print(f"❌ Error al consultar BigQuery: {e}")
    print("\nVerificá que:")
    print("  1. La BigQuery API esté habilitada")
    print("  2. El proyecto tenga facturación habilitada")
    print("  3. Tengas permisos de BigQuery User en el proyecto")

## **8) Configuración de LangSmith**

LangSmith es una plataforma de observabilidad para aplicaciones de LLM. Nos permite trackear todas las llamadas a los modelos, ver los prompts usados, y analizar el rendimiento.

**Pasos para configurar LangSmith:**

1. Andá a https://smith.langchain.com/ y creá una cuenta (gratis)
2. Creá un nuevo proyecto llamado `llmops-tutorial`
3. Andá a Settings → API Keys
4. Creá una nueva API key y copiala
5. Pegala en la celda siguiente

In [ ]:
import os

# CONFIGURÁ TU LANGSMITH API KEY ACÁ
LANGSMITH_API_KEY = "tu-langsmith-api-key"  # 👈 CAMBIAR ESTO

if LANGSMITH_API_KEY != "tu-langsmith-api-key":
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = "llmops-tutorial"

    print("✅ LangSmith configurado correctamente")
    print("   Proyecto: llmops-tutorial")
    print("   Endpoint: https://api.smith.langchain.com")
else:
    print("⚠️ Necesitás configurar LANGSMITH_API_KEY")
    print("   Si no querés usar LangSmith, podés saltear esta sección")

## **9) Configuración de OpenAI (Opcional)**

En la última sección del notebook principal, usamos OpenAI para demostrar el monitoring con LangSmith.

**Esto es OPCIONAL**. Si no querés usar OpenAI, podés:
- Saltear esta sección
- Usar Gemini en su lugar (ya configurado con GCP)

Para configurar OpenAI:
1. Andá a https://platform.openai.com/api-keys
2. Creá una API key
3. Pegala en la siguiente celda

In [ ]:
# OPCIONAL: Configurá tu OpenAI API Key
OPENAI_API_KEY = "tu-openai-api-key"  # 👈 CAMBIAR ESTO (opcional)

if OPENAI_API_KEY != "tu-openai-api-key":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✅ OpenAI API Key configurada")
else:
    print("ℹ️ OpenAI API Key no configurada (opcional)")
    print("   Podés usar Gemini en lugar de OpenAI para la demo de LangSmith")

## **10) Creación de Storage Bucket (Opcional)**

Si querés guardar artefactos intermedios del pipeline (checkpoints, logs, etc.), es útil tener un bucket de Cloud Storage.

**Esto es OPCIONAL** para el tutorial básico, pero recomendado para pipelines más complejos.

In [ ]:
# Configuramos el nombre del bucket
BUCKET_NAME = f"{PROJECT_ID}-llmops-artifacts"
BUCKET_URI = f"gs://{BUCKET_NAME}"

# Intentamos crear el bucket
try:
    !gsutil mb -p {PROJECT_ID} -l {LOCATION} {BUCKET_URI}
    print(f"✅ Bucket creado: {BUCKET_URI}")
except:
    print(f"ℹ️ El bucket {BUCKET_URI} ya existe o no se pudo crear")
    print("   Si ya existe, podés usarlo sin problemas")
    print("   Si no querés usar bucket, podés saltear esta sección")

## **11) Verificación final**

Vamos a hacer una verificación final de que todo está configurado correctamente antes de ejecutar el notebook principal.

In [ ]:
print("=" * 60)
print("VERIFICACIÓN FINAL DE CONFIGURACIÓN")
print("=" * 60)

checks = []

# Check 1: Proyecto configurado
if PROJECT_ID and PROJECT_ID != "tu-project-id":
    print("✅ Proyecto configurado:", PROJECT_ID)
    checks.append(True)
else:
    print("❌ Proyecto NO configurado")
    checks.append(False)

# Check 2: BigQuery funciona
try:
    test_query = "SELECT 1 as test"
    bq_client.query(test_query).result()
    print("✅ BigQuery funcionando")
    checks.append(True)
except:
    print("❌ BigQuery NO funciona")
    checks.append(False)

# Check 3: Vertex AI inicializado
try:
    from google.cloud import aiplatform
    print("✅ Vertex AI inicializado")
    checks.append(True)
except:
    print("❌ Vertex AI NO inicializado")
    checks.append(False)

# Check 4: LangSmith (opcional)
if os.getenv("LANGSMITH_API_KEY") and os.getenv("LANGSMITH_API_KEY") != "tu-langsmith-api-key":
    print("✅ LangSmith configurado")
else:
    print("ℹ️ LangSmith no configurado (opcional)")

# Check 5: OpenAI (opcional)
if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != "tu-openai-api-key":
    print("✅ OpenAI configurado")
else:
    print("ℹ️ OpenAI no configurado (opcional)")

print("\n" + "=" * 60)

if all(checks):
    print("\n🎉 TODO LISTO! Podés ejecutar el notebook principal.\n")
    print("Variables exportadas para usar en el notebook principal:")
    print(f"  - PROJECT_ID: {PROJECT_ID}")
    print(f"  - LOCATION: {LOCATION}")
    print(f"  - UID: {UID}")
    if 'BUCKET_URI' in locals():
        print(f"  - BUCKET_URI: {BUCKET_URI}")
else:
    print("\n⚠️ Hay algunos checks que fallaron.")
    print("Revisá los errores arriba antes de continuar.")

print("=" * 60)

## **12) Troubleshooting común**

### Error: "API not enabled"
```bash
# Verificá que todas las APIs estén habilitadas
gcloud services list --enabled --project=TU-PROJECT-ID
```

### Error: "Permission denied" en BigQuery
- Los datasets públicos son gratuitos y no necesitan permisos especiales
- Verificá que tu proyecto tenga facturación habilitada
- Verificá que tengas el rol "BigQuery User"

### Error: "Quota exceeded"
- Revisá tus cuotas en: https://console.cloud.google.com/iam-admin/quotas
- Podés solicitar aumentos de cuota si es necesario
- El tier gratuito tiene límites generosos para este tutorial

### El pipeline falla en Vertex AI
- Verificá los logs en la consola de Vertex AI
- Asegurate de que todas las APIs estén habilitadas
- Verificá que el PROJECT_ID esté configurado correctamente

## **13) Próximos pasos**

Si todos los checks pasaron exitosamente, estás listo para ejecutar el notebook principal: **`llmops_practica.ipynb`**

Antes de ejecutarlo, asegurate de:

1. **Actualizar el PROJECT_ID** en el notebook principal con el mismo valor que usaste acá
2. **Copiar las variables de entorno** de LangSmith y OpenAI (si las configuraste)
3. **Ejecutar las celdas en orden** y leer las explicaciones

**Recursos útiles:**
- Documentación de Vertex AI: https://cloud.google.com/vertex-ai/docs
- Documentación de BigQuery: https://cloud.google.com/bigquery/docs
- Kubeflow Pipelines: https://www.kubeflow.org/docs/components/pipelines/
- LangSmith: https://docs.smith.langchain.com/

**¡Éxitos con el pipeline de LLMOps!** 🚀

## **14) Exportar variables para el notebook principal**

In [ ]:
# Esta celda guarda las variables en un archivo para cargar en el notebook principal
import json

config = {
    "PROJECT_ID": PROJECT_ID,
    "LOCATION": LOCATION,
    "UID": UID,
}

if 'BUCKET_URI' in locals():
    config["BUCKET_URI"] = BUCKET_URI

with open('llmops_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Configuración guardada en: llmops_config.json")
print("\nPodés cargarla en el notebook principal con:")
print("""\nimport json
with open('llmops_config.json', 'r') as f:
    config = json.load(f)
PROJECT_ID = config['PROJECT_ID']
LOCATION = config['LOCATION']
""")